# Pipeline smoke test: loading + stitching + motion correction

Runs the loading/stitching functions in `caiman_wrapper.loading_tiff_files` and the motion-correction functions in `caiman_wrapper.motion_correction` end to end against a real uploaded session, to confirm the pipeline works before building anything further on top of it.

Uses the `caiman_wrapper` conda environment (real CaImAn from conda-forge + the editable-installed `caiman_wrapper` package).

In [ ]:
from caiman_wrapper.loading_tiff_files import create_output_directory, convert_files_for_caiman
from caiman_wrapper.motion_correction import (
    start_cluster,
    set_motion_correction_params,
    run_motion_correction,
    apply_shifts_to_ca,
)
import caiman as cm

## 1. Loading and stitching

`base_directory` points directly at the session folder (no `raw_data/` subfolder in this dataset). Each `TSeries-...` run under it is split into separate `Ch1` (reference) and `Ch2` (calcium) file sets, one multipage OME-TIFF per cycle.

In [ ]:
base_directory = '../data/2026_05_19_animal_1'
dest_directory = '../data/processed'

output_dir = create_output_directory(base_directory, dest_directory)
output_dir

In [ ]:
# This reads and stitches every TSeries run's Ch1/Ch2 files (930 files per
# channel across both runs here) and writes RefData.h5 / CaData.h5 -- expect
# this to take a while and use several GB of memory given the real data size.
ref_path, ca_path = convert_files_for_caiman(base_directory, output_dir, single_page=False)
ref_path, ca_path

## 2. Motion correction

Compute NoRMCorre shifts on the reference channel, then apply them to the calcium channel.

In [ ]:
c, dview, n_processes = start_cluster()
n_processes

In [ ]:
opts = set_motion_correction_params(fnames=[str(ref_path)])

In [ ]:
mc = run_motion_correction(opts, dview=dview)

In [ ]:
ref_moco_path, ca_moco_path = apply_shifts_to_ca(mc, str(ca_path), output_dir)
ref_moco_path, ca_moco_path

In [ ]:
cm.stop_server(dview=dview)